# Measuring bias and variance

Train the same kind of model on 200 different random training sets and see how its predictions vary.

In [ ]:
import warnings
import numpy as np
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
warnings.simplefilter("ignore")

f = lambda x: np.sin(1.5 * x) + 0.5 * x       # the true pattern (normally unknown)
NOISE = 0.5
xs = np.linspace(-2.8, 2.8, 200)

In [ ]:
def fits(degree, n_sets=200, n=40, seed=0):
    rng = np.random.default_rng(seed)
    preds = []
    for _ in range(n_sets):                       # a fresh training set each time
        X = rng.uniform(-3, 3, (n, 1))
        y = f(X).ravel() + rng.normal(0, NOISE, n)
        model = make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), LinearRegression())
        preds.append(model.fit(X, y).predict(xs.reshape(-1, 1)))
    return np.array(preds)                        # one row of predictions per training set

for d in [1, 3, 5, 7, 9, 11]:
    P = fits(d)
    bias2 = np.mean((P.mean(axis=0) - f(xs)) ** 2)    # how far the AVERAGE model is from the truth
    var = np.mean(P.var(axis=0))                      # how much the models disagree with each other
    print(f"degree {d:2d}: bias^2 {bias2:7.3f}  variance {var:8.3f}  expected error {bias2 + var + NOISE**2:8.3f}")

In [ ]:
P = fits(11)
fig = go.Figure([go.Scatter(x=xs, y=np.clip(p, -4, 4), mode="lines", opacity=0.4, showlegend=False) for p in P[:20]])
fig.add_scatter(x=xs, y=f(xs), mode="lines", name="truth", line=dict(color="black", dash="dash", width=3))
fig.update_layout(title="Degree 11 on 20 different training sets", yaxis_range=[-4, 4])
fig